# Day 046 Project: Trend Analysis Report

## What You're Building

A `TrendAnalyzer` pipeline that takes a daily time series, produces a full summary report, and saves a matplotlib chart showing the raw values and their 7-day rolling mean.

## Project Requirements

1. Generate or load a time series with at least 60 data points
2. Create a `TrendAnalyzer`, call `.load(df, 'date', 'value')`
3. Call `.summary()` and store the result as `report`
4. Print: n_periods, trend_direction, total_pct_change, start_date, end_date
5. Save a chart with the raw series + 7-day rolling mean to `trend_chart.png`
6. Run `_run_project_checks()` to verify

You run it, it prints a trend summary and saves a chart. That is the deliverable.

## Provided: All Implementations

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def make_sample_ts(n_days: int = 90, seed: int = 42) -> pd.DataFrame:
    """Return a reproducible daily time-series DataFrame for exercises."""
    rng   = np.random.default_rng(seed)
    dates = pd.date_range('2024-01-01', periods=n_days, freq='D')
    vals  = 1000.0 + (rng.standard_normal(n_days).cumsum() * 50)
    return pd.DataFrame({'date': dates.strftime('%Y-%m-%d'),
                         'value': vals.round(2)})


import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def parse_time_series(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])
    df = df.set_index(date_col).sort_index()
    return df

def date_features(df: pd.DataFrame) -> pd.DataFrame:
    idx = df.index
    return pd.DataFrame({
        'year':        idx.year,
        'month':       idx.month,
        'day':         idx.day,
        'day_of_week': idx.dayofweek,
        'quarter':     idx.quarter,
    }, index=idx)


def resample_series(series: pd.Series, freq: str, agg: str = 'sum') -> pd.Series:
    return series.resample(freq).agg(agg)

def multi_freq_summary(series: pd.Series) -> dict:
    return {
        'daily':  resample_series(series, 'D'),
        'weekly': resample_series(series, 'W'),
    }


def rolling_mean(series: pd.Series, window: int, min_periods: int = 1) -> pd.Series:
    return series.rolling(window=window, min_periods=min_periods).mean()

def rolling_stats(series: pd.Series, window: int) -> pd.DataFrame:
    r = series.rolling(window=window, min_periods=1)
    return pd.DataFrame({
        'mean': r.mean(),
        'std':  r.std(),
        'min':  r.min(),
        'max':  r.max(),
    })


def period_changes(series: pd.Series) -> pd.DataFrame:
    return pd.DataFrame({
        'value':      series,
        'change':     series.diff(),
        'pct_change': series.pct_change() * 100,
        'cumulative': series.cumsum(),
    })


class TrendAnalyzer:
    def __init__(self):
        self._series = None

    def load(self, df: pd.DataFrame, date_col: str, value_col: str) -> 'TrendAnalyzer':
        self._series = parse_time_series(df, date_col)[value_col].dropna()
        return self

    def summary(self) -> dict:
        s = self._series
        if s is None or len(s) == 0:
            return {}
        first, last = float(s.iloc[0]), float(s.iloc[-1])
        total_pct   = (last - first) / abs(first) * 100 if first != 0 else 0.0
        direction   = 'up' if total_pct > 1 else ('down' if total_pct < -1 else 'flat')
        return {
            'n_periods':        len(s),
            'start_date':       str(s.index[0].date()),
            'end_date':         str(s.index[-1].date()),
            'first_value':      round(first, 2),
            'last_value':       round(last, 2),
            'total_pct_change': round(total_pct, 2),
            'trend_direction':  direction,
            'weekly_avg':       resample_series(s, 'W', 'mean'),
            'rolling_mean_7d':  rolling_stats(s, 7)['mean'],
            'daily_changes':    period_changes(s),
        }

## Your Pipeline

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# TODO: generate or load data
# df = make_sample_ts(n_days=90, seed=7)

# TODO: create and load the analyzer
# ta = TrendAnalyzer()
# ta.load(df, 'date', 'value')
# report = ta.summary()

# TODO: print report
# print(f"Periods: {report['n_periods']}")
# print(f"Range:   {report['start_date']} → {report['end_date']}")
# print(f"Trend:   {report['trend_direction']}  ({report['total_pct_change']:+.2f}%)")

# TODO: save chart
# fig, ax = plt.subplots(figsize=(12, 4))
# report['rolling_mean_7d'].plot(ax=ax, label='7-day rolling mean', linewidth=2)
# ax.set_title('Trend Analysis')
# ax.legend()
# fig.savefig('trend_chart.png', bbox_inches='tight', dpi=100)
# plt.close('all')
# print('Chart saved: trend_chart.png')

## Checks

In [ ]:
import os

def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: report defined as dict
    try:
        assert 'report' in globals(), 'report not defined — call ta.summary()'
        assert isinstance(report, dict)
        passed += 1; print('\u2705 Check 1: report is a dict')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: required keys present
    try:
        for k in ('n_periods', 'trend_direction', 'total_pct_change',
                  'start_date', 'end_date', 'rolling_mean_7d'):
            assert k in report, f'missing key: {k!r}'
        passed += 1; print('\u2705 Check 2: report has all required keys')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: at least 60 periods
    try:
        n = report['n_periods']
        assert n >= 60, f'need >= 60 periods, got {n}'
        passed += 1; print(f'\u2705 Check 3: {n} periods (>= 60)')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: trend_direction is valid
    try:
        d = report['trend_direction']
        assert d in ('up', 'down', 'flat'), \
            f'trend_direction must be up/down/flat, got {d!r}'
        passed += 1; print(f'\u2705 Check 4: trend_direction={d!r}')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: chart file saved
    try:
        assert os.path.exists('trend_chart.png'), \
            'trend_chart.png not found — save with fig.savefig()'
        assert os.path.getsize('trend_chart.png') > 1000, \
            'trend_chart.png looks empty'
        passed += 1; print('\u2705 Check 5: trend_chart.png saved')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Try a monthly resample: `resample_series(report['rolling_mean_7d'], 'ME', 'mean')`
- Use `date_features()` to plot average value by day of week (bar chart)
- Add a second subplot showing daily `pct_change` to spot volatility spikes
- Load a real CSV from Day 45's pipeline and run `TrendAnalyzer` on it
- Use `ollama.chat` (Day 40 pattern) to narrate the trend summary in plain English